<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Trial/2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step.
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file.
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**

## 1. Basic Setup

In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/cvnd_coco"
CODE_DIR = "/content/code"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

print("✅ ROOT:", ROOT)
print("✅ CODE_DIR:", CODE_DIR)


Mounted at /content/drive
✅ ROOT: /content/drive/MyDrive/cvnd_coco
✅ CODE_DIR: /content/code


In [2]:
!pip -q install nltk pycocotools tqdm

import nltk
nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
import urllib.request, sys, os

BASE  = "https://raw.githubusercontent.com/pradhapmoorthi/CVND/Trial"
files = ["model.py", "data_loader.py", "vocabulary.py"]

# ✅ FRESH START: always re-download — never use a cached stale model.py
for f in files:
    dst = os.path.join(CODE_DIR, f)
    urllib.request.urlretrieve(f"{BASE}/{f}", dst)
    print(f"✅ Downloaded: {dst}")

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print("✅ sys.path[0]:", sys.path[0])


In [4]:
import os
import urllib.request
import zipfile

COCO_ROOT = "/content/cocoapi"
os.makedirs(COCO_ROOT, exist_ok=True)
os.makedirs(f"{COCO_ROOT}/images", exist_ok=True)
os.makedirs(f"{COCO_ROOT}/annotations", exist_ok=True)

TRAIN_IMG_URL = "http://images.cocodataset.org/zips/train2014.zip"
ANN_URL       = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"

TRAIN_ZIP = "/content/train2014.zip"
ANN_ZIP   = "/content/annotations_trainval2014.zip"

def download_if_missing(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print("✅ zip exists:", dst)
        return
    print("⬇️ downloading:", url)
    urllib.request.urlretrieve(url, dst)
    print("✅ downloaded:", dst)

def unzip_if_needed(zip_path, out_dir, expected_path=None):
    if expected_path and os.path.exists(expected_path):
        print("✅ already extracted:", expected_path)
        return
    print("📦 extracting:", zip_path, "->", out_dir)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print("✅ extracted")

download_if_missing(TRAIN_IMG_URL, TRAIN_ZIP)
download_if_missing(ANN_URL, ANN_ZIP)

# Extract train images into COCO_ROOT/images (creates train2014 folder)
unzip_if_needed(TRAIN_ZIP, f"{COCO_ROOT}/images", expected_path=f"{COCO_ROOT}/images/train2014")

# Extract annotations into COCO_ROOT (creates annotations folder with captions_train2014.json)
unzip_if_needed(ANN_ZIP, COCO_ROOT, expected_path=f"{COCO_ROOT}/annotations/captions_train2014.json")

print("\n✅ COCO structure check:")
!ls -l /content/cocoapi
!ls -l /content/cocoapi/images | head
!ls -l /content/cocoapi/annotations | grep captions | head

⬇️ downloading: http://images.cocodataset.org/zips/train2014.zip
✅ downloaded: /content/train2014.zip
⬇️ downloading: http://images.cocodataset.org/annotations/annotations_trainval2014.zip
✅ downloaded: /content/annotations_trainval2014.zip
📦 extracting: /content/train2014.zip -> /content/cocoapi/images
✅ extracted
📦 extracting: /content/annotations_trainval2014.zip -> /content/cocoapi
✅ extracted

✅ COCO structure check:
total 8
drwxr-xr-x 2 root root 4096 Apr 27 09:26 annotations
drwxr-xr-x 3 root root 4096 Apr 27 09:25 images
total 4656
drwxr-xr-x 2 root root 4763648 Apr 27 09:26 train2014
-rw-r--r-- 1 root root  66782097 Apr 27 09:26 captions_train2014.json
-rw-r--r-- 1 root root  32421077 Apr 27 09:26 captions_val2014.json


In [5]:
import os

train_img_dir = "/content/cocoapi/images/train2014"
cap_file = "/content/cocoapi/annotations/captions_train2014.json"

assert os.path.isdir(train_img_dir), f"Missing train image dir: {train_img_dir}"
assert os.path.isfile(cap_file), f"Missing captions file: {cap_file}"

print("✅ Paths OK")
print("Train images:", len(os.listdir(train_img_dir)))
print("Captions file:", cap_file)

✅ Paths OK
Train images: 82783
Captions file: /content/cocoapi/annotations/captions_train2014.json


In [ ]:
import glob, os

# ✅ FRESH START: remove ALL previous checkpoints before training
# This is the single most important step — every previous bad run
# left stale ckpt_best.pt / ckpt_epochN.pt files on Drive.
# The inference notebook auto-loads ckpt_best.pt, so a stale file
# from a collapsed run will always produce garbage captions.
ckpts = glob.glob(f"{ROOT}/ckpt_*.pt")
if ckpts:
    for ck in sorted(ckpts):
        os.remove(ck)
        print(f"🗑️  Deleted: {ck}")
    print(f"\n✅ Cleared {len(ckpts)} old checkpoint(s) — clean slate")
else:
    print("✅ No old checkpoints found — already clean")

print("\n🆕 Ready for a FRESH training run from randomly-initialised weights")


In [6]:
import torchvision.transforms as transforms
from data_loader import get_loader
import nltk
nltk.download('punkt_tab')

# ✅ FIX 1: Correct transform matching ResNet-50 ImageNet pretraining
# Resize shorter edge to 256 first, then centre-crop to 224x224.
# Using Resize((224,224)) directly squashes aspect ratio and degrades features.
transform = transforms.Compose([
    transforms.Resize(256),          # resize shorter edge to 256
    transforms.CenterCrop(224),      # centre-crop to 224x224 (matches ImageNet pretraining)
    transforms.RandomHorizontalFlip(),  # light augmentation for training
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

VOCAB_FILE = f"{ROOT}/vocab.pkl"

# ✅ FIX 2: Reduced batch_size 128→64
# Smaller batches produce less noisy attention gradients and
# prevent the attention weights from saturating early in training.
data_loader = get_loader(
    transform=transform,
    mode="train",
    batch_size=64,
    vocab_threshold=5,
    vocab_file=VOCAB_FILE,
    vocab_from_file=False,  # ✅ FIRST RUN: build vocab
    num_workers=2,
    cocoapi_loc="/content"
)

print("✅ Dataset size:", len(data_loader.dataset))
print("✅ Vocab size:", len(data_loader.dataset.vocab))
print("✅ Vocab saved/used at:", VOCAB_FILE)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


loading annotations into memory...
Done (t=0.64s)
creating index...
index created!
[0/414113] Tokenizing captions...
[100000/414113] Tokenizing captions...
[200000/414113] Tokenizing captions...
[300000/414113] Tokenizing captions...
[400000/414113] Tokenizing captions...
loading annotations into memory...
Done (t=0.62s)
creating index...
index created!
Obtaining caption lengths...


100%|██████████| 414113/414113 [00:18<00:00, 22315.85it/s]


✅ Dataset size: 414113
✅ Vocab size: 8852
✅ Vocab saved/used at: /content/drive/MyDrive/cvnd_coco/vocab.pkl


In [7]:
import torch
import torch.nn as nn

vocab = data_loader.dataset.vocab

# Try common pad token strings first (in case it exists in other runs)
pad_idx = None
for tok in ["<pad>", "<PAD>", "[PAD]", "<null>", "<NULL>"]:
    if hasattr(vocab, "word2idx") and tok in vocab.word2idx:
        pad_idx = vocab.word2idx[tok]
        print("✅ Found pad token:", tok, "->", pad_idx)
        break

# If no pad token exists, infer from batch statistics
if pad_idx is None:
    _, captions = next(iter(data_loader))
    vals, counts = torch.unique(captions, return_counts=True)
    pad_idx = vals[counts.argmax()].item()
    print("✅ No pad token in vocab; inferred pad_idx (most frequent id) =", pad_idx)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
print("✅ criterion.ignore_index =", pad_idx)

✅ No pad token in vocab; inferred pad_idx (most frequent id) = 3
✅ criterion.ignore_index = 3


In [8]:
!ls

annotations_trainval2014.zip  cocoapi  code  drive  sample_data  train2014.zip


In [9]:
import torch
import importlib, model as _model_mod
importlib.reload(_model_mod)          # reload from disk — picks up latest model.py
from model import EncoderCNN, DecoderRNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", device)

# Hyperparameters — must match across training AND inference
encoded_image_size = 7
attention_dim      = 512
embed_size         = 256
hidden_size        = 512
num_layers         = 1
vocab_size         = len(data_loader.dataset.vocab)

# ✅ FRESH START: always create new model objects — never reuse from a prior cell run
encoder = EncoderCNN(encoded_image_size=encoded_image_size).to(device)
decoder = DecoderRNN(
    attention_dim = attention_dim,
    embed_size    = embed_size,
    hidden_size   = hidden_size,
    vocab_size    = vocab_size,
    encoder_dim   = encoder.encoder_dim,
    dropout       = 0.5
).to(device)

# ✅ FRESH START: explicitly reset all decoder learnable weights
# Protects against Colab session reuse where model object persists in memory
for module in decoder.modules():
    if hasattr(module, 'reset_parameters'):
        module.reset_parameters()
print("✅ Decoder weights randomly re-initialised")

# Encoder stays frozen (pretrained ResNet-50 weights) — intentional
frozen = sum(1 for p in encoder.parameters() if not p.requires_grad)
total  = sum(1 for p in encoder.parameters())
print(f"✅ Encoder: {frozen}/{total} params frozen (pretrained ResNet-50)")
print(f"✅ vocab_size={vocab_size}  embed={embed_size}  "
      f"hidden={hidden_size}  attn={attention_dim}")


✅ Device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 226MB/s]


In [10]:
# ✅ FRESH START: optimizer always created from scratch — no load_state_dict
optimizer = torch.optim.Adam(
    list(filter(lambda p: p.requires_grad, encoder.parameters())) +
    list(decoder.parameters()),
    lr=3e-4      # must always be 3e-4 at the start of a fresh run
)

num_epochs = 30

# ✅ CosineAnnealingLR replaces StepLR
# StepLR(step_size=8, gamma=0.5) over 30 epochs: LR dies at epoch 16 (~1e-5)
# CosineAnnealingLR over 30 epochs: LR stays above 1e-4 until epoch 20,
#   above 1e-5 until epoch 27 — giving the model sustained learning signal
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max   = num_epochs,
    eta_min = 1e-6         # floor — never goes below this
)

# ✅ Hard guard: if LR is not 3e-4 here, a stale optimizer is loaded
assert abs(optimizer.param_groups[0]['lr'] - 3e-4) < 1e-9, (
    f"\u274c LR is {optimizer.param_groups[0]['lr']:.2e}, expected 3e-4. "
    f"Re-run the model-init cell to create a fresh optimizer."
)
print(f"\u2705 Optimizer lr : {optimizer.param_groups[0]['lr']:.2e}  (3e-4 confirmed)")
print(f"\u2705 Scheduler    : CosineAnnealingLR  T_max={num_epochs}  eta_min=1e-6")
print(f"\u2705 num_epochs   : {num_epochs}")

# LR schedule preview
import torch
_sched_preview = torch.optim.lr_scheduler.CosineAnnealingLR(
    torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=3e-4),
    T_max=num_epochs, eta_min=1e-6
)
print("\nLR schedule preview:")
for ep in [1, 5, 10, 15, 20, 25, 30]:
    _lr = 3e-4  # simulate
    _tmp = torch.optim.lr_scheduler.CosineAnnealingLR(
        torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=3e-4),
        T_max=num_epochs, eta_min=1e-6
    )
    for _ in range(ep): _tmp.step()
    print(f"  Epoch {ep:2d}: {_tmp.get_last_lr()[0]:.2e}")


✅ Model initialized correctly
encoder_dim: 2048 vocab_size: 8852
optimizer lr: 0.0003


In [11]:
import time, os

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    _smoother = SmoothingFunction().method4
    nltk_ok = True
except Exception:
    nltk_ok = False

# num_epochs defined in Cell 14 (= 30)
log_every  = 100
grad_clip  = 1.0
best_bleu  = 0.0
best_ckpt  = os.path.join(ROOT, "ckpt_best.pt")

start_idx = vocab.word2idx["<start>"]
end_idx   = vocab.word2idx["<end>"]


def evaluate_bleu(loader, limit=300):
    """
    BLEU-4 estimate on up to `limit` training batches.

    Prints two diagnostic numbers every epoch:
      captioned : images where the model produced at least one word
      empty     : images where model predicted <end> immediately

    Expected progression for a healthy run:
      Epoch  1 : ~10/300  captioned  (mostly empty — normal)
      Epoch  5 : ~100/300 captioned
      Epoch 10 : ~220/300 captioned
      Epoch 20 : ~290/300 captioned
    If captioned stays <20 beyond epoch 8 → training has collapsed.
    """
    if not nltk_ok:
        return None
    encoder.eval()
    decoder.eval()
    refs, hyps, empty_count = [], [], 0
    with torch.no_grad():
        for step, (images, captions) in enumerate(loader):
            if step >= limit:
                break
            enc_out = encoder(images[:1].to(device))
            ids = decoder.sample(enc_out, start_idx, end_idx, max_len=20)
            if not ids:
                empty_count += 1
                continue
            hyps.append(ids)
            gt = [t.item() for t in captions[0]
                  if t.item() not in (start_idx, end_idx, pad_idx)]
            refs.append([gt])
    print(f"     [{len(hyps)}/{limit} captioned | {empty_count} empty "
          f"— should reach 200+ by epoch 10]")
    if not refs:
        return 0.0
    return corpus_bleu(refs, hyps, smoothing_function=_smoother)


for epoch in range(1, num_epochs + 1):
    encoder.train()
    decoder.train()
    running_loss = 0.0
    t0 = time.time()
    print(f"\n===== Epoch {epoch}/{num_epochs}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e} =====")

    for step, (images, captions) in enumerate(data_loader):
        images   = images.to(device)
        captions = captions.to(device)

        features        = encoder(images)
        outputs, alphas = decoder(features, captions)
        targets         = captions[:, 1:].reshape(-1)
        if outputs.dim() == 3:
            outputs = outputs.reshape(-1, outputs.size(-1))

        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(decoder.parameters()) +
            list(filter(lambda p: p.requires_grad, encoder.parameters())),
            max_norm=grad_clip
        )
        optimizer.step()
        running_loss += loss.item()

        if (step + 1) % log_every == 0:
            avg = running_loss / log_every
            print(f"  Step {step+1}/{len(data_loader)}  "
                  f"Loss={avg:.4f}  {time.time()-t0:.1f}s")
            running_loss = 0.0
            t0 = time.time()

    scheduler.step()

    ckpt_path = os.path.join(ROOT, f"ckpt_epoch{epoch}.pt")
    torch.save({
        "epoch":                epoch,
        "encoder_state_dict":   encoder.state_dict(),
        "decoder_state_dict":   decoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "embed_size":           embed_size,
        "hidden_size":          hidden_size,
        "attention_dim":        attention_dim,
        "num_layers":           num_layers,
        "vocab_file":           VOCAB_FILE,
    }, ckpt_path)
    print(f"\u2705 Saved: {ckpt_path}")

    bleu = evaluate_bleu(data_loader, limit=300)
    if bleu is not None:
        print(f"   BLEU-4: {bleu:.4f}")
        if bleu > best_bleu:
            best_bleu = bleu
            torch.save({
                "epoch":              epoch,
                "encoder_state_dict": encoder.state_dict(),
                "decoder_state_dict": decoder.state_dict(),
                "embed_size":         embed_size,
                "hidden_size":        hidden_size,
                "attention_dim":      attention_dim,
                "vocab_file":         VOCAB_FILE,
            }, best_ckpt)
            print(f"   \u2b50 New best BLEU! Saved to: {best_ckpt}")

print("\n\U0001f389 Training complete")
print(f"   Best BLEU-4: {best_bleu:.4f}  \u2192  {best_ckpt}")



===== Epoch 1/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch1.pt
   BLEU-4 estimate: 0.0000

===== Epoch 2/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch2.pt
   BLEU-4 estimate: 0.0000

===== Epoch 3/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch3.pt
   BLEU-4 estimate: 0.0070
   ⭐ New best BLEU! Saved to: /content/drive/MyDrive/cvnd_coco/ckpt_best.pt

===== Epoch 4/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch4.pt
   BLEU-4 estimate: 0.0000

===== Epoch 5/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch5.pt
   BLEU-4 estimate: 0.0000

===== Epoch 6/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch6.pt
   BLEU-4 estimate: 0.0000

===== Epoch 7/25  lr=3.00e-04 =====
✅ Saved checkpoint: /content/drive/MyDrive/cvnd_coco/ckpt_epoch7.pt
   BLEU-4 estimate: 0.0000


<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here.

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.